# PySpark AI Functions Starter Notebook

Use PySpark AI Functions on customer-review dataset to build a transformation workflow.

## What You'll Do
1. Set shared defaults with `aifunc.default_conf`, including `api_type="chat_completions"`.
2. Load a public review sample and inspect rating and review-length patterns before enrichment.
3. Walk through all nine Fabric AI Functions on a Spark DataFrame.
4. Build a compact dashboard and review common PySpark configuration options.

## Before You Start
- **Runtime**: Fabric Runtime 1.3 or later.
- **Built-in dependency**: PySpark AI Functions are available in Fabric runtime without the extra `openai` install required by the pandas notebook.
- **Cost control**: Start with `SAMPLE_ROWS`, then scale up after you are happy with output quality.

| Resource | Link |
|---|---|
| AI Functions overview | [aka.ms/ai-functions](https://aka.ms/ai-functions) |
| PySpark configuration docs | [PySpark AI Function Config Doc](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration) |
| Model catalog | [aka.ms/fabric-ai-models](https://aka.ms/fabric-ai-models) |
| More starter notebook examples | [aka.ms/fabric-aifunctions-starter-notebooks](https://aka.ms/fabric-aifunctions-starter-notebooks) |
| More eval notebook examples | [aka.ms/fabric-aifunctions-eval-notebooks](https://aka.ms/fabric-aifunctions-eval-notebooks) |

## 1. Imports and Setup

In [ ]:
from textwrap import dedent

import matplotlib.pyplot as plt
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from datasets import load_dataset
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, desc, get_json_object, length

SAMPLE_ROWS = 20

In [ ]:
# Shared chart styling and color palette
plt.style.use("seaborn-v0_8-whitegrid")

FABRIC_BLUE = "#4C78A8"
FABRIC_TEAL = "#72B7B2"
FABRIC_ORANGE = "#F58518"
FABRIC_RED = "#E45756"
FABRIC_GRAY = "#9D9DA1"
SENTIMENT_COLORS = {
    "positive": "#54A24B",
    "negative": FABRIC_RED,
    "neutral": FABRIC_GRAY,
    "mixed": FABRIC_ORANGE,
}
PRIORITY_COLORS = {
    "high": FABRIC_RED,
    "medium": FABRIC_ORANGE,
    "low": FABRIC_TEAL,
}

### 1.1 Shared defaults and per-function tuning

PySpark AI Functions support two tuning scopes:
- `aifunc.default_conf.set_*` methods for shared session defaults such as API type, model deployments, and temperature
- Per-function parameters such as `concurrency=200` when a single transformation needs focused throughput control

Use `aifunc.default_conf.set_*` for workflow-wide settings. This walkthrough keeps the API and model defaults fixed across the main steps, then shows `concurrency` later as a per-function parameter. Unlike the model defaults, `concurrency` is not configured through `aifunc.default_conf`.

In [ ]:
# Shared defaults for the walkthrough.
aifunc.default_conf.set_deployment_name("gpt-4.1-mini")  # Text model used across examples.
aifunc.default_conf.set_embedding_deployment_name("text-embedding-ada-002")  # Embedding model used by ai.embed and ai.similarity.
aifunc.default_conf.set_api_type("chat_completions")  # Use the chat completions API surface.
aifunc.default_conf.set_temperature(0.0)  # Deterministic outputs for repeatable results.

display(
    pd.DataFrame(
        {
            "setting": [
                "api_type",
                "deployment_name",
                "embedding_deployment_name",
                "temperature",
                "default_concurrency_per_worker",
            ],
            "value": [
                "chat_completions",
                "gpt-4.1-mini",
                "text-embedding-ada-002",
                0.0,
                200,
            ],
            "why_it_matters": [
                "Chooses the model API surface for notebook-wide calls",
                "Controls which text model runs the AI functions",
                "Controls which embedding model powers ai.embed and ai.similarity",
                "Keeps outputs deterministic for a starter workflow",
                "PySpark uses 200 concurrent requests per worker node by default for each AI function call",
            ],
        }
    )
)

### 1.2 Common PySpark configuration methods

| Global method | Why you would use it |
|---|---|
| `set_api_type(...)` | Switch between `responses` and `chat_completions`. |
| `set_deployment_name(...)` | Choose the text model deployment. |
| `set_embedding_deployment_name(...)` | Choose the embedding model deployment. |
| `set_temperature(...)` | Adjust randomness for standard models. |
| `set_reasoning_effort(...)` | Tune gpt-5 reasoning depth. |
| `set_verbosity(...)` | Control gpt-5 output length. |
| `set_URL(...)` and `set_subscription_key(...)` | Point AI Functions at your own endpoint. |
| `reset_*()` | Return to the built-in Fabric defaults. |

Use per-function parameters such as `concurrency` when a specific transformation needs different parallelism. `concurrency` is configured per function call rather than through `aifunc.default_conf`. The default is 200 per worker node, so a cluster with 4 workers can issue up to 800 requests in parallel for a single transformation. Because Spark evaluation is lazy, this notebook stores reusable outputs in named DataFrames and uses `.cache()` when later cells will reuse the same result. When you want to chain another AI function onto an AI-generated column, use `.cache().select("*")`: `.cache()` marks the result for reuse, the first action materializes that cached plan, and `.select("*")` returns a fresh DataFrame object whose `.ai` accessor binds to the current schema. For complete parameter details, see [PySpark AI Function Config Doc](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

## 2. Load data and profile the sample

Use the Amazon Reviews 2023 sample to keep iteration fast while preserving a realistic review-analysis workflow on a Spark DataFrame.

In [ ]:
reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Sports_and_Outdoors",
    split="full",
    streaming=True,
    trust_remote_code=True,
)

raw_pdf = pd.DataFrame(list(reviews.take(1000)))
sample_pdf = raw_pdf.sample(n=min(SAMPLE_ROWS, len(raw_pdf)), random_state=2026).reset_index(drop=True)
sample_pdf = sample_pdf.rename(columns={"text": "review_text", "title": "review_title"})
sample_pdf["review_title"] = sample_pdf["review_title"].fillna("").astype(str)
sample_pdf["review_text"] = sample_pdf["review_text"].fillna("").astype(str)

spark_df = spark.createDataFrame(sample_pdf)
reviews_df = (
    spark_df.select("review_title", "review_text", "rating")
    .withColumn("review_char_count", length(col("review_text")))
    .cache()
)
row_count = reviews_df.count()

display(
    pd.DataFrame(
        {
            "metric": ["rows_loaded", "columns", "dataset_config"],
            "value": [row_count, ", ".join(reviews_df.columns), "raw_review_Sports_and_Outdoors"],
        }
    )
)
display(reviews_df.limit(10))

In [ ]:
row_count = reviews_df.count()
text_stats = reviews_df.select(avg(col("review_char_count")).alias("avg_review_char_count")).first()
median_review_char_count = reviews_df.approxQuantile("review_char_count", [0.5], 0.0)[0] if row_count else 0
rating_dist = reviews_df.groupBy("rating").count().orderBy("rating").toPandas()
review_length_pd = reviews_df.select("review_char_count").toPandas()

eda_summary = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "avg_review_char_count",
            "median_review_char_count",
            "distinct_ratings",
        ],
        "value": [
            row_count,
            round(text_stats["avg_review_char_count"], 1) if text_stats["avg_review_char_count"] else 0,
            round(median_review_char_count, 1) if median_review_char_count else 0,
            reviews_df.select("rating").distinct().count(),
        ],
    }
)
display(eda_summary)
display(rating_dist)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(rating_dist["rating"].astype(str), rating_dist["count"], color=FABRIC_BLUE, edgecolor="white")
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("count")

review_length_pd["review_char_count"].plot.hist(
    ax=axes[1],
    bins=min(12, max(5, len(review_length_pd) // 2)),
    color=FABRIC_TEAL,
    edgecolor="white",
)
axes[1].axvline(
    review_length_pd["review_char_count"].median(),
    color=FABRIC_ORANGE,
    linestyle="--",
    linewidth=2,
    label="median",
)
axes[1].set_title("Review length distribution")
axes[1].set_xlabel("characters")
axes[1].set_ylabel("count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Walk through the AI Functions

This example workflow moves from signal detection -> enrichment -> action: start with sentiment and topic labels, extract structured facts, condense and clean the text, add semantic signals, and finish with support-ready outputs.

### 3.1 `ai.analyze_sentiment`

Start with sentiment so every downstream step has a first-pass triage signal.

In [ ]:
sentiment_df = reviews_df.ai.analyze_sentiment(input_col="review_text", output_col="sentiment").cache()

display(sentiment_df.select("review_text", "rating", "sentiment").limit(10))

In [ ]:
sentiment_dist = sentiment_df.groupBy("sentiment").count().toPandas()
sentiment_lookup = dict(zip(sentiment_dist["sentiment"], sentiment_dist["count"]))
sentiment_order = ["positive", "neutral", "mixed", "negative"]
sentiment_plot = pd.DataFrame(
    {
        "sentiment": sentiment_order,
        "count": [sentiment_lookup.get(value, 0) for value in sentiment_order],
    }
)
display(sentiment_plot)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    sentiment_plot["sentiment"],
    sentiment_plot["count"],
    color=[SENTIMENT_COLORS.get(value, FABRIC_BLUE) for value in sentiment_plot["sentiment"]],
    edgecolor="white",
)
ax.set_title("Sentiment distribution")
ax.set_xlabel("sentiment")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

### 3.2 `ai.classify`

Add a business topic so each review is easier to route to the right owner.

In [ ]:
topic_df = reviews_df.ai.classify(
    labels=[
        "product_quality",
        "shipping_delivery",
        "customer_service",
        "price_value",
        "product_description",
    ],
    input_col="review_text",
    output_col="topic",
).cache()

display(topic_df.select("review_text", "topic").limit(10))

In [ ]:
topic_dist = topic_df.groupBy("topic").count().orderBy(desc("count")).toPandas()

display(topic_dist)

fig, ax = plt.subplots(figsize=(8, 4.5))
topic_plot = topic_dist.sort_values("count", ascending=True)
ax.barh(topic_plot["topic"], topic_plot["count"], color=FABRIC_BLUE, edgecolor="white")
ax.set_title("Top classified topics")
ax.set_xlabel("count")
ax.set_ylabel("topic")
plt.tight_layout()
plt.show()

### 3.3 `ai.extract`

Extract product, brand, and issue details so the triage view contains concrete facts, not just labels.

In [ ]:
extract_df = reviews_df.ai.extract(labels=["product", "brand", "issue"], input_col="review_text").cache()

display(extract_df.select("review_text", "product", "brand", "issue").limit(10))

In [ ]:
entity_summary = pd.DataFrame(
    {
        "entity": ["product", "brand", "issue"],
        "rows_with_values": [
            extract_df.filter(col("product").isNotNull() & (length(col("product").cast("string")) > 2)).count(),
            extract_df.filter(col("brand").isNotNull() & (length(col("brand").cast("string")) > 2)).count(),
            extract_df.filter(col("issue").isNotNull() & (length(col("issue").cast("string")) > 2)).count(),
        ],
    }
).sort_values("rows_with_values", ascending=True)
display(entity_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(entity_summary["entity"], entity_summary["rows_with_values"], color=FABRIC_TEAL, edgecolor="white")
ax.set_title("Entity coverage across the sample")
ax.set_xlabel("rows with at least one extracted value")
ax.set_ylabel("entity")
plt.tight_layout()
plt.show()

### 3.4 `ai.summarize`

Condense longer reviews into a format that is easier to scan during triage.

In [ ]:
summary_df = reviews_df.ai.summarize(input_col="review_text", output_col="summary").cache().select("*")

# The first action below materializes the cached summary result for reuse.
display(summary_df.select("review_text", "summary").limit(10))

In [ ]:
compression_stats = (
    summary_df.select(
        avg(length(col("review_text"))).alias("avg_original_chars"),
        avg(length(col("summary"))).alias("avg_summary_chars"),
    )
    .toPandas()
)
compression_stats["compression_percent"] = (
    1 - (compression_stats["avg_summary_chars"] / compression_stats["avg_original_chars"])
) * 100

display(compression_stats.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["original", "summary"],
    [compression_stats.loc[0, "avg_original_chars"], compression_stats.loc[0, "avg_summary_chars"]],
    color=[FABRIC_BLUE, FABRIC_TEAL],
    edgecolor="white",
)
ax.set_title("Average review length vs summary length")
ax.set_ylabel("characters")
plt.tight_layout()
plt.show()

### 3.5 `ai.translate`

Translate summaries so the same workflow can support multilingual review processes. The summarization step above uses `.cache().select("*")` so Spark can reuse the computed summaries and the next `.ai` call resolves against the refreshed schema that includes `summary`.

In [ ]:
translated_summary_df = summary_df.ai.translate(
    to_lang="spanish",
    input_col="summary",
    output_col="summary_spanish",
)

display(translated_summary_df.select("summary", "summary_spanish").limit(10))

### 3.6 `ai.fix_grammar`

Clean up raw review text when you need something easier to share with business users.

In [ ]:
grammar_df = reviews_df.ai.fix_grammar(input_col="review_text", output_col="review_corrected")

display(grammar_df.select("review_text", "review_corrected").limit(10))

### 3.7 `ai.embed`

Generate embeddings for semantic search, clustering, and other vector-based workflows.

In [ ]:
embedding_df = reviews_df.ai.embed(input_col="review_text", output_col="embedding").cache()

sample_embedding_row = embedding_df.select("embedding").where(col("embedding").isNotNull()).limit(1).collect()
embedding_dimension = len(sample_embedding_row[0]["embedding"]) if sample_embedding_row else 0
embedding_preview = sample_embedding_row[0]["embedding"][:5] if sample_embedding_row else []

embedding_summary = pd.DataFrame(
    {
        "metric": ["rows_with_embeddings", "embedding_dimension", "first_vector_preview"],
        "value": [
            embedding_df.filter(col("embedding").isNotNull()).count(),
            embedding_dimension,
            str(embedding_preview),
        ],
    }
)
display(embedding_summary)

### 3.8 `ai.similarity`

Rank reviews against a focused business question to surface the most relevant rows first.

In [ ]:
query = "durability and product quality issues"
similarity_df = reviews_df.ai.similarity(input_col="review_text", other=query, output_col="similarity_score").cache()

display(similarity_df.select("review_text", "similarity_score").orderBy(desc("similarity_score")).limit(10))

### 3.9 `ai.generate_response`

Turn each review into a structured support recommendation that is easy to operationalize. Start by keeping the raw JSON output in its own DataFrame so you can inspect the generated payload. The cached raw result then supports the rest of the same response workflow: first parse the JSON fields into columns for downstream PySpark transformations, then review `ai.stats` on the original `raw_response_df` without rerunning the model. This step keeps the response workflow self-contained and makes the PySpark throughput knob explicit by setting `concurrency=200` on a single transformation. That value matches the default per worker node, so a cluster with 4 workers can issue up to 800 requests in parallel for this step.

In [ ]:
response_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "response_suggestion",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string"},
                "response_type": {
                    "type": "string",
                    "enum": ["thank_you", "apology", "follow_up", "no_response_needed"],
                },
                "priority": {"type": "string", "enum": ["high", "medium", "low"]},
                "suggested_response": {"type": "string"},
            },
            "required": ["reason", "response_type", "priority", "suggested_response"],
            "additionalProperties": False,
        },
    },
}

display(
    pd.DataFrame(
        {
            "response_schema_fields": ["reason", "response_type", "priority", "suggested_response"],
        }
    )
)

In [ ]:
RESPONSE_PROMPT = dedent(
    """
    You are a helpful customer support assistant.

    Review:
    <review>{review_text}</review>

    Return a warm and practical response to the customer.
    - Infer the customer's sentiment and likely topic from the review before drafting the response.
    - If the review sounds negative, apologize and include a concrete next step.
    - If the review sounds positive, thank the customer and reinforce value.
    - Keep the suggested response to 2-3 sentences.
    """
).strip()

# Cache and materialize the raw AI output once so the preview, stats, and parsed fields reuse the same responses.
raw_response_df = (
    reviews_df.ai.generate_response(
        prompt=RESPONSE_PROMPT,
        is_prompt_template=True,
        output_col="response_json",
        response_format=response_schema,
        concurrency=200,
    )
    .cache()
)
_ = raw_response_df.count()

display(raw_response_df.select("review_text", "response_json").limit(10))

Parse the cached JSON payload into structured columns that power the downstream dashboard and quick aggregations. Because `response_df` is derived from the cached `raw_response_df`, you can preview the extracted fields here and still inspect `ai.stats` on the original AI function output in the next section.

In [ ]:
# Parse the JSON payload into columns used by the dashboard and cache the result for later aggregations.
response_df = (
    raw_response_df
    .withColumn("response_reason", get_json_object(col("response_json"), "$.reason"))
    .withColumn("response_type", get_json_object(col("response_json"), "$.response_type"))
    .withColumn("priority", get_json_object(col("response_json"), "$.priority"))
    .withColumn("suggested_response", get_json_object(col("response_json"), "$.suggested_response"))
    .cache()
)

display(
    response_df.select("review_text", "response_reason", "response_type", "priority", "suggested_response").limit(10)
)

display(response_df.groupBy("response_type").count().orderBy(desc("count")))
display(response_df.groupBy("priority").count().orderBy(desc("count")))

### 3.10 `ai.stats`

`ai.stats` is not an AI function itself. It is a helper that summarizes the AI Functions calls already run on `raw_response_df`, so you can review usage details such as request counts, success or failure totals, and token usage on the original AI function output dataframe.

In [ ]:
display(raw_response_df.ai.stats)

## 4. Build a triage dashboard

The dashboard below combines the cached outputs from the earlier sections, including the parsed response fields derived from the cached raw response result, so Spark does not need to rerun every AI transformation for each display.

In [ ]:
pipeline_summary = pd.DataFrame(
    {
        "metric": [
            "rows_processed",
            "unique_sentiments",
            "unique_topics",
            "negative_reviews",
            "high_priority_items",
        ],
        "value": [
            reviews_df.count(),
            sentiment_df.select("sentiment").distinct().count(),
            topic_df.select("topic").distinct().count(),
            sentiment_df.filter(col("sentiment") == "negative").count(),
            response_df.filter(col("priority") == "high").count(),
        ],
    }
)
display(pipeline_summary)

display(
    response_df.filter(col("priority") == "high")
    .select("review_text", "response_type", "priority", "suggested_response")
    .limit(10)
)

In [ ]:
sentiment_pd = sentiment_df.groupBy("sentiment").count().toPandas()
sentiment_lookup = dict(zip(sentiment_pd["sentiment"], sentiment_pd["count"]))
topic_pd = topic_df.groupBy("topic").count().orderBy(desc("count")).limit(8).toPandas()
response_pd = response_df.groupBy("response_type").count().orderBy(desc("count")).toPandas()
priority_pd = response_df.groupBy("priority").count().toPandas()
priority_lookup = dict(zip(priority_pd["priority"], priority_pd["count"]))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sentiment_order = ["positive", "neutral", "mixed", "negative"]
sentiment_values = [sentiment_lookup.get(value, 0) for value in sentiment_order]
axes[0, 0].bar(
    sentiment_order,
    sentiment_values,
    color=[SENTIMENT_COLORS.get(value, FABRIC_BLUE) for value in sentiment_order],
    edgecolor="white",
)
axes[0, 0].set_title("Sentiment")
axes[0, 0].set_ylabel("count")

topic_plot = topic_pd.sort_values("count", ascending=True)
axes[0, 1].barh(topic_plot["topic"], topic_plot["count"], color=FABRIC_BLUE, edgecolor="white")
axes[0, 1].set_title("Top topics")
axes[0, 1].set_xlabel("count")

axes[1, 0].bar(response_pd["response_type"], response_pd["count"], color=FABRIC_TEAL, edgecolor="white")
axes[1, 0].set_title("Response type")
axes[1, 0].set_ylabel("count")
axes[1, 0].tick_params(axis="x", rotation=20)

priority_order = ["high", "medium", "low"]
priority_values = [priority_lookup.get(value, 0) for value in priority_order]
axes[1, 1].bar(
    priority_order,
    priority_values,
    color=[PRIORITY_COLORS.get(value, FABRIC_BLUE) for value in priority_order],
    edgecolor="white",
)
axes[1, 1].set_title("Priority")
axes[1, 1].set_ylabel("count")

plt.tight_layout()
plt.show()

## 5. Interpreting results and next steps

### How to use this workflow
- Use **sentiment + topic** to identify where customer pain points are clustering.
- Use **extract** to pull substrings from text fields such as products, brands, and issues into downstream triage.
- Use **summary + translation + grammar fixes** to make reviews easier to read and share.
- Use **similarity + structured response suggestions** to prioritize repeated issues and accelerate support follow-up.

### Next steps
- Increase `SAMPLE_ROWS` once the notebook quality and cost look right.
- Replace the sample dataset with your own Delta table or Spark table.
- Keep this PySpark pattern when the dataset grows beyond what you want to keep in memory locally.
- Use the pandas starter notebook when you want faster iteration on smaller samples.
- Use eval notebooks when you want systematic quality measurement instead of a starter walkthrough.

### Learn more
- Starter notebooks: [aka.ms/fabric-aifunctions-starter-notebooks](https://aka.ms/fabric-aifunctions-starter-notebooks)
- Eval notebooks: [aka.ms/fabric-aifunctions-eval-notebooks](https://aka.ms/fabric-aifunctions-eval-notebooks)
- AI Functions overview: [aka.ms/ai-functions](https://aka.ms/ai-functions)
- [PySpark AI Function Config Doc](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- Model catalog: [aka.ms/fabric-ai-models](https://aka.ms/fabric-ai-models)